In [41]:
import pandas as pd
import numpy as np
import re
from collections import Counter

# Pfade
INPUT_CSV = "cards.csv"
OUTPUT_CSV = "cards_clean.csv"
OVERWRITE_INPUT = False

def clean_money_string(s: str) -> str:
    if pd.isna(s):
        return s
    s = str(s).strip()
    s = s.replace('\u00a0', '').replace('$','').replace('€','').replace('£','')
    if ',' in s and '.' in s:
        s = s.replace(',', '')
    else:
        if ',' in s and s.count(',') == 1 and not re.search(r'\d,\d{3}', s):
            s = s.replace(',', '.')
        else:
            s = s.replace(',', '')
    s = re.sub(r'[^\d\.-]', '', s)
    if s in ['', '-', '.', '-.']:
        return np.nan
    return s

def to_numeric_safe(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    s = s.replace({'None': np.nan, 'nan': np.nan, 'NaN': np.nan, '': np.nan, 'N/A': np.nan, 'n/a': np.nan, 'Not Available': np.nan})
    s_cleaned = s.map(lambda x: clean_money_string(x) if pd.notna(x) else np.nan)
    return pd.to_numeric(s_cleaned, errors='coerce')

def first_nonnull(vals):
    for v in vals:
        if pd.notna(v):
            return v
    return np.nan

def mode_or_first(vals):
    vals_list = [v for v in vals if pd.notna(v)]
    if not vals_list:
        return np.nan
    try:
        most_common = Counter(vals_list).most_common(1)
        return most_common[0][0]
    except Exception:
        return vals_list[0]



In [42]:
print("Lade CSV...")
df = pd.read_csv(INPUT_CSV, low_memory=False)
print("Input shape:", df.shape)

Lade CSV...
Input shape: (40792, 19)


In [43]:
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan})


In [44]:
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])




In [45]:
numeric_columns_guess = ['price', 'volatility', 'attack', 'defense', 'rank']
for col in numeric_columns_guess:
    if col in df.columns:
        if col == 'volatility':
            df[col] = df[col].astype(str).str.replace('%','',regex=False).replace({'nan':np.nan})
        df[col] = to_numeric_safe(df[col])

In [46]:
if 'set_release' in df.columns:
    df['set_release_parsed'] = pd.to_datetime(df['set_release'], errors='coerce')

In [47]:
num_total = len(df)
num_price_na = df['price'].isna().sum() if 'price' in df.columns else 0
print(f"Total rows: {num_total}; rows with price NaN after cleaning: {num_price_na}")

Total rows: 40792; rows with price NaN after cleaning: 2659


In [48]:
dup_key_cols = ['name', 'description', 'type', 'sub_type', 'attribute', 'attack', 'defense']
set_fields = ['set_id', 'set_name', 'set_release', 'rarity', 'rank']

agg_dict = { 'price':'mean',
             'volatility':'mean',
             'index': first_nonnull,
             'index_market': first_nonnull,
             'join_id': first_nonnull }

for col in set_fields:
    if col in df.columns:
        agg_dict[col] = lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0]

for col in df.columns:
    if col not in dup_key_cols + list(agg_dict.keys()) and col != 'set_release_parsed':
        agg_dict[col] = mode_or_first

df_grouped = df.groupby(dup_key_cols, as_index=False).agg(agg_dict)

In [49]:
if 'price' in df_grouped.columns:
    df_grouped['price'] = df_grouped['price'].round(4)

for col in ['attack', 'defense', 'rank']:
    if col in df_grouped.columns:
        ser = df_grouped[col]
        non_null = ser.dropna()
        if len(non_null) > 0 and non_null.apply(float.is_integer).all():
            df_grouped[col] = ser.astype('Int64')
        else:
            df_grouped[col] = ser.astype('float')

if 'set_release_parsed' in df_grouped.columns:
    df_grouped['set_release'] = pd.to_datetime(df_grouped['set_release_parsed'], errors='coerce').dt.strftime('%Y-%m-%d')
    df_grouped = df_grouped.drop(columns=['set_release_parsed'])

In [50]:
print("Original rows:", num_total)
print("Rows after grouping (one row per card type):", len(df_grouped))
print(df_grouped.head(5).to_string(index=False))

Original rows: 40792
Rows after grouping (one row per card type): 8217
                     name                                                                                                                                                                                                  description    type             sub_type attribute  attack  defense  price  volatility  index  index_market     join_id     set_id                                          set_name set_release   rarity  rank             name_official
           3-Hump Lacooda                                                                                                  If there are 3 face-up "3-Hump Lacooda" cards on your side of the field, Tribute 2 of them to draw 3 cards. MONSTER       [Beast／Effect]     EARTH     500     1500 0.1650         NaN   6032         24804  DR2-EN183C    AST-070                                 ANCIENT SANCTUARY  2004-06-01 C Common     3            3-Hump Lacooda
4-Starred Ladybug of Do

In [51]:
df_grouped['price_missing'] = df_grouped['price'].isna()
num_missing_price = df_grouped['price_missing'].sum()
print(f"\nKarten mit fehlendem Preis: {num_missing_price}")
print("Erste 10 Karten ohne Preis:")
print(df_grouped[df_grouped['price_missing']].head(10)[['name','price']].to_string(index=False))


Karten mit fehlendem Preis: 113
Erste 10 Karten ohne Preis:
                           name  price
                     Ally Salvo    NaN
  Ally of Justice Thousand Arms    NaN
Ally of Justice Unknown Crusher    NaN
                     Amarylease    NaN
                      Amaterasu    NaN
            Anotherverse Dragon    NaN
         Anotherverse Gluttonia    NaN
           Anotherverse Solaria    NaN
                  Attack Gainer    NaN
                       Bite Bug    NaN


In [52]:
print("\n--- Fehlende Werte pro Spalte ---")
missing_counts = df_grouped.isna().sum()
print(missing_counts[missing_counts > 0])


--- Fehlende Werte pro Spalte ---
price           113
volatility     8217
set_release       3
dtype: int64


In [53]:
numeric_cols = df_grouped.select_dtypes(include=['float', 'int']).columns.tolist()
print("\nNumerische Spalten:", numeric_cols)

print("\n--- Ausreißer nach IQR-Methode ---")
for col in numeric_cols:
    if df_grouped[col].isna().all():
        continue
    Q1 = df_grouped[col].quantile(0.25)
    Q3 = df_grouped[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df_grouped[(df_grouped[col] < lower) | (df_grouped[col] > upper)]
    print(f"{col}: {len(outliers)} Ausreißer")
    if len(outliers) > 0:
        print(outliers[[col,'name']].head(5))


Numerische Spalten: ['attack', 'defense', 'price', 'volatility', 'index', 'index_market', 'rank']

--- Ausreißer nach IQR-Methode ---
attack: 29 Ausreißer
      attack                                      name
85      4500               Alba System Dogmatikalamity
827     4500     Blue-Eyes Alternative Ultimate Dragon
838     4500                 Blue-Eyes Ultimate Dragon
1066    4500                  Chaos Ancient Gear Giant
1298    4500  Crimson Nova Trinity the Dark Cubic Lord
defense: 12 Ausreißer
      defense                           name
2008     5000           Dragon Master Knight
2009     5000            Dragon Master Lords
2094     5000  Drytron Meteonis DA Draconids
2125     5000        Dystopia the Despondent
2634     5000             Five-Headed Dragon
price: 1266 Ausreißer
      price                              name
7    4.0967        A-Team: Trap Disposal Unit
34   9.4000                     Abyss Soldier
37  19.8033  Abyssrhine, the Atlantean Spirit
39   6.5633     

In [54]:
df_grouped.to_csv(OUTPUT_CSV, index=False)
if OVERWRITE_INPUT:
    df_grouped.to_csv(INPUT_CSV, index=False)
print(f"\nBereinigte CSV gespeichert nach: {OUTPUT_CSV}")
print("Fertig.")


Bereinigte CSV gespeichert nach: cards_clean.csv
Fertig.
